# Chat with a code or literature project

Choose a project, ask questions, inspect the exact evidence, and continue or
resume the same conversation.

## 1. Prepare QAChat

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Choose the project

Normally, you only need to edit these settings:

- **`PROJECT_KIND`** chooses **code** or **literature**.
- **`PROJECT_DIRECTORY`** chooses your original source folder.
- Leave **`PROJECT_DIRECTORY = None`** to use `example_repos/code_repo` or
  `example_repos/literature_repo`, depending on `PROJECT_KIND`.
- **`ENABLE_TRACE`** records the complete ordered execution for inspection.


In [ ]:
PROJECT_KIND = "literature"  # "code" or "literature"
PROJECT_DIRECTORY = None  # "/path/to/my/source-project"
ENABLE_TRACE = True

In [ ]:
if PROJECT_KIND not in {"code", "literature"}:
    raise ValueError("PROJECT_KIND must be `code` or `literature`.")


In [ ]:
if PROJECT_DIRECTORY is None:
    example_name = "code_repo" if PROJECT_KIND == "code" else "literature_repo"
    source_directory = (
        root / "example_repos" / example_name
    ).resolve(strict=True)
    print(f"Included example source: {source_directory}")
else:
    source_directory = (
        Path(PROJECT_DIRECTORY).expanduser().resolve(strict=True)
    )
    print(f"Original source: {source_directory}")


QAChat never writes project settings or indexes into the original source folder.
It mirrors the selected files into a managed workspace under **sandbox/**.
For example, `example_repos/code_repo` becomes
`sandbox/project_code_repo/code_repo`, with `indexes/`, `logs/`,
`profile.ini`, and `source_snapshot.json` beside the copied source folder.


## 3. Open or resume QAChat

On each open, QAChat compares SHA-256 source manifests, synchronizes only
added, removed, or modified source files into the sandbox, and then checks
whether the retrieval index still matches both the copied source and the
indexing settings. A current index is reused; otherwise it is rebuilt.


Every project, including the bundled example, uses the same sandbox workflow.
The original source folder is never modified.


In [ ]:
from qachat import prepare_project_workspace


project = prepare_project_workspace(
    source_directory,
    PROJECT_KIND,
    root / "sandbox",
)

project_directory = project.root
profile_path = project.root / "profile.ini"

print(f"QAChat workspace: {project.root}")
print(f"Indexed source snapshot: {project.source}")
print(f"Project settings: {profile_path}")


In [ ]:
index_path = project.indexes / "retrieval.json"


### Prepare progress reporting

Import the notebook display tools and the QAChat progress API. If a previous
QAChat session exists, close it before opening the project again.


In [ ]:
from html import escape
from IPython.display import HTML, display
from qachat import ProgressEvent, RAGSession, TraceConfig


if "session" in globals():
    session.close()


### Create the notebook progress display

The display is updated in place instead of printing a new line for every progress
event. `format_duration()` converts seconds into readable times such as `2m 15s`.


In [ ]:
progress_display = display(HTML(""), display_id=True)


def format_duration(seconds: float | None) -> str:
    if seconds is None:
        return "--"

    seconds = max(0, round(seconds))
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)

    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


### Define how progress is shown

QAChat sends `ProgressEvent` objects while preparing the project. This callback
turns those events into a live progress bar.

**Important:** `Stage ETA` estimates the remaining time for the **current stage only**
(for example, chunking or embedding). It is **not** an estimate of the remaining time
for the complete indexing workflow. The percentage is the overall workflow progress.


In [ ]:
def show_progress(event: ProgressEvent) -> None:
    width = 28
    percent = max(0.0, min(100.0, event.percent))
    filled = round(width * percent / 100.0)
    bar = "█" * filled + "░" * (width - filled)

    details = []
    if event.current is not None and event.total is not None:
        details.append(f"{event.current:,}/{event.total:,}")
    if event.remaining is not None:
        details.append(f"{event.remaining:,} remaining")
    if event.elapsed_seconds is not None:
        details.append(f"elapsed {format_duration(event.elapsed_seconds)}")
    if event.eta_seconds is not None:
        details.append(f"Stage ETA {format_duration(event.eta_seconds)}")

    detail_text = "  |  ".join(details)
    lines = [
        f"[{bar}] {percent:5.1f}%  {event.message}",
    ]
    if detail_text:
        lines.append(detail_text)

    progress_display.update(
        HTML(f"<pre style='margin:0'>{escape(chr(10).join(lines))}</pre>")
    )


### Open the QAChat session

This is the cell that actually checks the saved index and prepares QAChat. If the
project is unchanged, the existing index is reused. If indexed files or indexing
settings changed, QAChat rebuilds the index and reports progress through the callback
above. With `ENABLE_TRACE = True`, the session also records exact prompts, token
inputs, input/output token counts, retrieval decisions, per-call timings, settings,
and failures in `logs/traces`.


In [ ]:
session = RAGSession(
    project.root,
    progress_callback=show_progress,
    trace=TraceConfig(enabled=ENABLE_TRACE, mode="forensic"),
)
if session.trace.enabled:
    print(f"Readable trace: {session.trace.readable_path}")


In [ ]:
if session.turns:
    print("Resumed a previous conversation.")
    print(f"Restored turns: {len(session.turns)}")
    print(f"Last question: {session.turns[-1].question}")
elif index_path.is_file() and session.index_status == "reused":
    print("Opened an existing project and reused its index.")
else:
    print("Opened the project and built its index.")

changes = session.source_changes
print(f"Index status: {session.index_status}")
print(f"Backend: {session.backend}")
print(f"Source snapshot: {session.project.source}")

if session.index_status == "rebuilt":
    print(
        "Source changes: "
        f"+{len(changes['added'])} "
        f"~{len(changes['modified'])} "
        f"-{len(changes['removed'])}"
    )
    if changes["configuration_changed"]:
        print("Index settings are new or changed.")


## 4. Ask a question

Edit **QUESTION**, then run the following answer cell. QAChat immediately
saves the question, answer, evidence, and settings.

In [ ]:
DEFAULT_QUESTIONS = {
    "code": "How does this project convert Celsius to Kelvin?",
    "literature": "What did the toy sleep study observe?",
}

QUESTION = DEFAULT_QUESTIONS[session.project.profile.kind]

print(f"Question: {QUESTION}")


In [ ]:
answer = session.ask(QUESTION)

print(f"Answer: {answer}")


## 5. Inspect the evidence

The latest turn contains the exact source chunks given to the LLM. Their
numbers match the citations requested in the answer.

In [ ]:
latest_turn = session.turns[-1]

for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")
    print(item.text)
    print()


## 6. Keep asking questions

Edit **NEXT_QUESTION** and run the next cell with Shift+Enter. Repeat that
same cell for every new question. Keep the session open so each answer can
use the earlier dialogue as conversation history.

In [ ]:
NEXT_QUESTION = "Can you explain that in more detail?"

print(f"Next question: {NEXT_QUESTION}")

In [ ]:
next_answer = session.ask(NEXT_QUESTION)
latest_turn = session.turns[-1]

print(f"Answer: {next_answer}")


In [ ]:
print("Evidence:")
for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")


## 7. Find the saved conversation and settings

**profile.ini** contains the reusable project settings.


In [ ]:
profile_path = session.project.root / "profile.ini"

print(f"Project settings: {profile_path}")


**conversation.json** contains every question, answer, evidence set, and
parameter snapshot in the active conversation.


In [ ]:
conversation_path = session.project.logs / "conversation.json"

print(f"Active conversation: {conversation_path}")


In [ ]:
print(f"Reusable indexes: {session.project.indexes}")


In [ ]:
import json


print(json.dumps(latest_turn.parameters, indent=2, sort_keys=True))

## 8. Optionally start a new conversation

Set this option to **True** and run the next cell once. QAChat renames the
active log with a UTC timestamp, clears the conversation from memory, and
keeps the reusable indexes.

In [ ]:
START_NEW_CONVERSATION = True

if START_NEW_CONVERSATION:
    archived_path = session.reset()
    print(f"Archived conversation: {archived_path}")
    print(f"Active turns: {len(session.turns)}")

## 9. Close QAChat

Closing releases the models without removing indexes or conversations.

In [ ]:
session.close()

print("QAChat closed safely.")